# Web Agent Action Prediction — Monster Pipeline (Qwen3-32B / 80GB)

**최종 갱신: 2026-05-13 (Monster Mode v2)**

### 🚀 주요 특징
- **Base Model**: `unsloth/Qwen3-32B-unsloth-bnb-4bit` (Dense SOTA)
- **Optimization**: 80GB VRAM 풀가동 (DPO BS=16, Inference BS=128)
- **One-Click Setup**: 소스 코드림 셀에서 직접 생성하여 업로드 번거로움 제거
- **Thinking Mode**: Qwen3의 내장 사고 과정을 활용한 정밀한 UI Grounding

### 🛠️ 실행 순서
1. **GPU 확인**: A100 80GB 또는 H100 권장
2. **패키지 및 가속 커널 설치**: Unsloth + Flash Attention 2 + Qwen3 Kernels
3. **원클릭 코드 생성**: `src/*.py` 파일 자동 생성
4. **데이터 업로드**: `data/` 폴더에 CSV 파일 업로드
5. **OOF 학습 및 앙상블 추론**

## 1. GPU 확인

In [ ]:
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'GPU: {gpu_name} | VRAM: {total_vram:.2f} GB')
    
    if total_vram > 70:
        print('🔥 80GB Monster Mode 활성화 촀건 충족!')
    elif total_vram > 30:
        print('✅ 40GB A100 확인 (안정적인 학습 가능)')
    else:
        print('⚠️ VRAM이 부족할 수 있습니다. src/train.py의 배치림 낮촬야 할 수 있습니다.')
    
    !nvidia-smi
else:
    print('❌ GPU림 확인할 수 없습니다.')

## 2. 패키직 및 가속 커널 설치
A100/H100 하드웨어 가속을 위해 필수 커널들을 설치합니다. (약 3-5분 소요)

In [ ]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps unsloth_zoo
!pip install --no-deps "trl>=0.18.2,<=0.24.0" peft accelerate bitsandbytes
!pip install pandas tqdm scikit-learn lxml sentence-transformers bs4

print('토치오(torchao) 및 누락 의존성 패치...')
!pip install "torchao>=0.16.0" "datasets>=3.4.1,<4.0.0" "transformers<5.0.0"
!pip install cut_cross_entropy hf_transfer msgspec tyro

print('가속 커널 빌드 및 설치 중 (A100 80GB 최적화)...')
!pip install flash-attn --no-build-isolation
!pip install causal-conv1d flash-linear-attention mamba-ssm

print('설치 완료')

## 3. 원클릭 코드 생성 (One-Click Source Gen)
업로드 없이 `src/` 폴더 내의 모든 최적화된 소스 코드림 생성합니다.

In [ ]:
import os
os.makedirs('/content/src', exist_ok=True)
os.makedirs('/content/data', exist_ok=True)
os.makedirs('/content/artifacts', exist_ok=True)

preprocess_py = r'''# -*- coding: utf-8 -*-
import pandas as pd
import json
import re
import random
from collections import Counter
from typing import Any

CONSISTENCY_DEBUG = Counter()


# ─────────────────────────────────────────────
# 기본 파싱 유틸리티
# ─────────────────────────────────────────────

def parse_attrs_str(attrs_str: str) -> dict:
    result = {}
    if attrs_str is None or pd.isna(attrs_str):
        return result
    attrs_text = str(attrs_str).strip()
    if not attrs_text:
        return result
    pattern = re.compile(r'(?:^|\s*\|\s*)([A-Za-z0-9_:-]+)\s*=\s*(.*?)(?=\s*\|\s*[A-Za-z0-9_:-]+\s*=|$)')
    for match in pattern.finditer(attrs_text):
        key = match.group(1).strip()
        value = match.group(2).strip()
        if key:
            result[key] = value
    return result


def get_history_signature(history_str):
    """History의 op 시퀀스를 시그니처로 사용하여 워크플로우 분기 구별."""
    if pd.isna(history_str) or not str(history_str).strip():
        return "START"
    ops = re.findall(r'->\s*(CLICK|TYPE|SELECT)', str(history_str))
    if not ops:
        return "STEP_" + str(len(re.findall(r'Step \d+:', str(history_str))) + 1)
    return ",".join(ops)


# ─────────────────────────────────────────────
# Value 추출
# ─────────────────────────────────────────────

def extract_value_from_task(task: str, op: str, attrs: str) -> str:
    """task 문장에서 TYPE/SELECT에 필요한 value를 추출한다.

    우선순위:
    1. SELECT: options= 중 task에 등장하는 옵션 (exact → fuzzy, 동적 임계값)
    2. 따옴표 패턴
    3. TYPE: label/placeholder 기반 뒤따르는 청크 → task 말미 청크
    4. 날짜 regex (date 필드로 확인된 경우에만)
    5. 빈 문자열
    """
    op = str(op or 'CLICK').upper()
    if op == 'CLICK':
        return ""

    task_str  = "" if task is None or pd.isna(task) else str(task)
    attrs_str = "" if attrs is None or pd.isna(attrs) else str(attrs)

    try:
        if op == 'SELECT':
            options = _parse_options(attrs_str)
            task_lower = task_str.lower()
            for opt in options:
                if opt.lower() in task_lower:
                    return opt
            best_opt, best_score = None, 0.0
            task_words = _token_set(task_lower)
            for opt in options:
                opt_words = _token_set(opt)
                n_words = len(opt_words)
                score = len(opt_words & task_words) / max(n_words, 1)
                if score > best_score:
                    best_score, best_opt = score, opt
            # 동적 임계값: 단어 1개짜리 옵션은 0.3, 3개 이상은 0.6
            if best_opt:
                n = len(_token_set(best_opt))
                threshold = 0.3 if n <= 1 else (0.6 if n >= 3 else 0.5)
                if best_score >= threshold:
                    return best_opt

        quoted = re.search(r'["\'"]([^"\']+)["\'"]', task_str)
        if quoted:
            return quoted.group(1).strip()

        if op == 'TYPE':
            attrs_dict = parse_attrs_str(attrs_str)
            field_labels = [
                attrs_dict.get('label', ''),
                attrs_dict.get('placeholder', ''),
                attrs_dict.get('aria-label', ''),
                attrs_dict.get('name', ''),
            ]
            for field_label in [x for x in field_labels if x]:
                label_pattern = re.escape(str(field_label).strip())
                match = re.search(label_pattern + r'\s*(?:is|as|to|:|-)??\s*([^,.;\n]+)', task_str, flags=re.IGNORECASE)
                if match:
                    value = match.group(1).strip().strip('"\'')
                    if value:
                        return value

        # 날짜 regex: date 타입 필드로 확인된 경우에만 반환 (다중 포맷 지원)
        _DATE_PATTERNS = [
            r'\d{4}-\d{2}-\d{2}',                      # YYYY-MM-DD
            r'\d{1,2}/\d{1,2}/\d{4}',                  # MM/DD/YYYY or M/D/YYYY
            r'\d{1,2}-\d{1,2}-\d{4}',                  # MM-DD-YYYY
            r'(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\.?\s+\d{1,2},?\s+\d{4}',
        ]
        attrs_dict = parse_attrs_str(attrs_str)
        field_type = attrs_dict.get('type', '').lower()
        field_label_all = ' '.join([
            attrs_dict.get('label', ''), attrs_dict.get('placeholder', ''),
            attrs_dict.get('aria-label', ''), attrs_dict.get('name', ''),
        ]).lower()
        if 'date' in field_type or 'date' in field_label_all:
            for _pat in _DATE_PATTERNS:
                date_match = re.search(_pat, task_str, re.IGNORECASE)
                if date_match:
                    return date_match.group(0)
    except Exception:
        return ""

    return ""


# ─────────────────────────────────────────────
# 내부 유틸리티
# ─────────────────────────────────────────────

def _candidate_label(c):
    label = str(c.get('text', '')).strip().lower()
    if not label:
        attrs_dict = parse_attrs_str(str(c.get('attrs', '')))
        # SeeAct empty-text fallback chain
        label = str(
            attrs_dict.get('aria-label', '') or
            attrs_dict.get('placeholder', '') or
            attrs_dict.get('title', '') or
            attrs_dict.get('role', '') or
            attrs_dict.get('label', '') or
            attrs_dict.get('name', '')
        ).strip().lower()
    return label


def _parse_options(attrs: str):
    if attrs is None or pd.isna(attrs):
        return []
    opts_str = parse_attrs_str(str(attrs)).get('options', '')
    if not opts_str:
        return []
    return [o.strip() for o in opts_str.split(' / ') if o.strip()]


def _token_set(text: str):
    return set(re.findall(r'\w+', str(text).lower()))


def _candidate_match_score(candidate: dict[str, Any], task: str) -> float:
    task_lower = str(task).lower()
    attrs_dict = parse_attrs_str(str(candidate.get('attrs', '')))
    fields = [
        candidate.get('text', ''),
        attrs_dict.get('label', ''),
        attrs_dict.get('placeholder', ''),
        attrs_dict.get('aria-label', ''),
        attrs_dict.get('name', ''),
    ]
    score = 0.0
    for word in re.findall(r'\w+', ' '.join(str(f) for f in fields).lower()):
        if len(word) > 2 and word in task_lower:
            score += 1.0
    label_words = _token_set(' '.join(str(f) for f in fields))
    task_words  = _token_set(task)
    if label_words:
        score += len(label_words & task_words) / max(len(label_words), 1)
    return score


def _best_candidate(pool, task: str):
    if not pool:
        return None
    return max(pool, key=lambda c: (_candidate_match_score(c, task), str(c.get('candidate_id', ''))))


def _best_option_for_task(options, task: str, current_value: str = ""):
    if not options:
        return ""

    value_norm = str(current_value).strip().lower()
    for opt in options:
        if str(opt).strip().lower() == value_norm:
            return opt

    value_words = _token_set(current_value)
    best_opt, best_score = None, 0.0
    if value_words:
        for opt in options:
            opt_words = _token_set(opt)
            score = len(opt_words & value_words) / max(len(opt_words), 1)
            if score > best_score:
                best_score, best_opt = score, opt
        if best_opt and best_score >= 0.5:
            return best_opt

    task_words = _token_set(task)
    best_opt, best_score = options[0], -1.0
    for opt in options:
        opt_words = _token_set(opt)
        score = len(opt_words & task_words) / max(len(opt_words), 1)
        if score > best_score:
            best_score, best_opt = score, opt
    return best_opt or options[0]


# ─────────────────────────────────────────────
# Consistency Guard
# ─────────────────────────────────────────────

def reset_consistency_debug():
    CONSISTENCY_DEBUG.clear()


def get_consistency_debug():
    return dict(CONSISTENCY_DEBUG)


def enforce_consistency(pred, candidates):
    """최종 가드레일: op/tag/value 일관성을 강제한다.

    선택적 키 `_task`는 수리 점수 계산에 사용되며, 반환 전에 제거된다.
    """
    task = str(pred.get('_task', '')) if isinstance(pred, dict) else ''
    pred = dict(pred or {})
    pred.pop('_task', None)
    pred.pop('_row', None)

    valid_ops = {'CLICK', 'TYPE', 'SELECT'}
    op = str(pred.get('op', 'CLICK')).upper()
    if op not in valid_ops:
        op = 'CLICK'
        CONSISTENCY_DEBUG['bad_op_to_click'] += 1

    cand_map  = {str(c.get('candidate_id', '')): c for c in candidates}
    target_id = str(pred.get('target_id', ''))
    value     = "" if pd.isna(pred.get('value', '')) else str(pred.get('value', ''))

    if target_id not in cand_map:
        fb = fallback_rule_based({'task': task}, candidates)
        op, target_id, value = fb['op'], str(fb['target_id']), str(fb.get('value', ''))
        CONSISTENCY_DEBUG['invalid_target_repaired'] += 1
        # fallback 결과도 cand_map에 없으면 첫 번째 후보로 안전하게 대체
        if target_id not in cand_map and candidates:
            target_id = str(candidates[0].get('candidate_id', ''))
            CONSISTENCY_DEBUG['fallback_target_defaulted'] += 1

    chosen = cand_map.get(str(target_id))
    tag    = str(chosen.get('tag', '')).lower() if chosen else ''

    if op == 'CLICK' and tag == 'select':
        extracted = extract_value_from_task(task, 'SELECT', str(chosen.get('attrs', ''))) if chosen else ''
        if extracted:
            op, value = 'SELECT', extracted
            CONSISTENCY_DEBUG['click_select_upgraded'] += 1
        else:
            replacement = _best_candidate(
                [c for c in candidates if str(c.get('tag', '')).lower() in {'button', 'a'}], task
            )
            if replacement:
                target_id = str(replacement.get('candidate_id', ''))
                chosen    = replacement
                tag       = str(chosen.get('tag', '')).lower()
                CONSISTENCY_DEBUG['click_select_target_switched'] += 1

    if op == 'SELECT':
        options     = _parse_options(str(chosen.get('attrs', ''))) if chosen else []
        fixed_value = _best_option_for_task(options, task, value)
        if not fixed_value:
            # options 파싱 실패 시 LLM 원본값 유지, 없으면 task에서 직접 추출
            fixed_value = value or extract_value_from_task(task, 'SELECT', str(chosen.get('attrs', '')) if chosen else '')
            if fixed_value:
                CONSISTENCY_DEBUG['select_value_kept_raw'] += 1
            elif options:
                # 모든 방법 실패 시 첫 번째 옵션 사용 (빈 value 방지)
                fixed_value = options[0]
                CONSISTENCY_DEBUG['select_value_defaulted_first_option'] += 1
        elif fixed_value != value:
            CONSISTENCY_DEBUG['select_value_repaired'] += 1
        value = fixed_value

    if op == 'CLICK':
        if value:
            CONSISTENCY_DEBUG['click_value_cleared'] += 1
        value = ""

    return {'op': op, 'target_id': str(target_id), 'value': value}


# ─────────────────────────────────────────────
# Rule-based Fallback
# ─────────────────────────────────────────────

def _infer_op_from_task(task: str) -> str:
    task_lower = str(task).lower()
    if re.search(r'\btype\b|\benter\b|\binput\b|fill(?:\s+in)?\b|write\b', task_lower):
        return "TYPE"
    if re.search(r'\bselect\b|\bchoose\b|\bpick\b|\bset\b', task_lower):
        return "SELECT"
    if re.search(r'\bclick\b|\bpress\b|\bsubmit\b|\bopen\b|\bgo\b', task_lower):
        return "CLICK"
    return "CLICK"


def _candidate_relevance_score(candidate: dict[str, Any], task: str, op: str) -> float:
    tag   = str(candidate.get('tag', '')).lower()
    attrs = str(candidate.get('attrs', ''))
    score = _candidate_match_score(candidate, task)

    if op == "TYPE":
        score += 2.0 if tag in {'input', 'textarea'} else -1.0
    elif op == "SELECT":
        score += 2.0 if tag == 'select' else -1.0
        options    = _parse_options(attrs)
        task_words = _token_set(task)
        for opt in options:
            opt_words = _token_set(opt)
            if opt.lower() in str(task).lower():
                score += 2.0
            elif opt_words:
                score += len(opt_words & task_words) / max(len(opt_words), 1)
    elif op == "CLICK":
        score += 1.0 if tag in {'button', 'a'} else 0.0

    return score


def fallback_rule_based(row, candidates):
    """LLM 실패 시 rule 기반으로 op/target_id/value를 결정한다."""
    task      = str(row['task'])
    op        = _infer_op_from_task(task)
    target_id = candidates[0].get('candidate_id', '') if candidates else ""
    value     = ""

    def best_candidate(pool):
        if not pool:
            return None
        return max(
            pool,
            key=lambda c: (_candidate_relevance_score(c, task, op), str(c.get('candidate_id', '')))
        )

    if op == "TYPE":
        pool = [c for c in candidates if str(c.get('tag', '')).lower() in {'input', 'textarea'}]
    elif op == "SELECT":
        pool = [c for c in candidates if str(c.get('tag', '')).lower() == 'select']
    elif op == "CLICK":
        pool = [c for c in candidates if str(c.get('tag', '')).lower() in {'button', 'a'}]
    else:
        pool = []

    chosen = best_candidate(pool) or best_candidate(candidates)
    if chosen:
        target_id = chosen.get('candidate_id', '')
        value     = extract_value_from_task(task, op, str(chosen.get('attrs', '')))

    if op == "CLICK":
        value = ""

    return {'op': op, 'target_id': target_id, 'value': value}


# ─────────────────────────────────────────────
# HTML 타입 감지 및 Workflow 컨텍스트 파싱
# ─────────────────────────────────────────────

def is_workflow_html(html_str) -> bool:
    """Workflow형 HTML 여부 판별."""
    if not isinstance(html_str, str):
        return False
    return ("workflow-context" in html_str) or ("completed-fields" in html_str)


def detect_html_type(row) -> str:
    """행의 cleaned_html을 보고 'workflow' 또는 'real_web' 반환."""
    return "workflow" if is_workflow_html(str(row.get("cleaned_html", ""))) else "real_web"


def extract_workflow_context(html_str) -> dict:
    """Workflow HTML에서 현재 단계 및 완료된 필드 정보를 추출한다.

    반환 예시:
        {"current_step": "6", "total_steps": "7", "completed_fields": ["Name", "Date"]}
    """
    result = {"current_step": "", "total_steps": "", "completed_fields": []}
    if not isinstance(html_str, str):
        return result
    step_m = re.search(r"current step\s+(\d+)\s+of\s+(\d+)", html_str, re.IGNORECASE)
    if step_m:
        result["current_step"] = step_m.group(1)
        result["total_steps"]  = step_m.group(2)
    completed_m = re.search(r"Completed:\s*([^\n<]+)", html_str, re.IGNORECASE)
    if completed_m:
        fields = [f.strip() for f in completed_m.group(1).split(",") if f.strip()]
        result["completed_fields"] = fields
    return result


# ─────────────────────────────────────────────
# HTML 컨텍스트 추출 (real_web 전용)
# ─────────────────────────────────────────────

def _find_element_in_soup(soup, candidate):
    tag   = candidate.get("tag", "") or ""
    attrs = parse_attrs_str(str(candidate.get("attrs", "")))
    text  = str(candidate.get("text", "")).strip()
    for key in ("id", "name", "placeholder", "aria-label"):
        val = attrs.get(key)
        if not val:
            continue
        el = soup.find(tag, attrs={key: val}) if tag else soup.find(attrs={key: val})
        if el:
            return el
    if text and tag:
        for el in soup.find_all(tag):
            if el.get_text(strip=True)[:60] == text[:60]:
                return el
    return None


def get_html_context(soup, candidate) -> str:
    if soup is None:
        return ""
    try:
        el = _find_element_in_soup(soup, candidate)
        if not el:
            return ""
        parts = []
        el_id = el.get("id")
        if el_id:
            label = soup.find("label", {"for": el_id})
            if label:
                t = label.get_text(strip=True)[:50]
                if t:
                    parts.append(f"label:{t}")
        if not parts:
            lby = el.get("aria-labelledby")
            if lby:
                ref = soup.find(id=lby)
                if ref:
                    t = ref.get_text(strip=True)[:50]
                    if t:
                        parts.append(f"label:{t}")
        if not parts:
            prev = el.find_previous_sibling()
            if prev:
                t = prev.get_text(strip=True)[:40]
                if t:
                    parts.append(f"prev:{t}")
        parent = el.parent
        if parent and parent.name not in ("html", "body", "[document]", None):
            p_role = parent.get("aria-label") or parent.get("role") or ""
            p_tag  = parent.name
            if p_tag in ("form", "fieldset", "section", "nav", "header", "main", "footer"):
                parts.append(f"in:<{p_tag}{'[' + p_role[:25] + ']' if p_role else ''}>")

        # 자식 요소 (최대 3개 — 버튼 내부 텍스트, select 옵션 등)
        children = el.find_all(recursive=False)[:3]
        child_texts = []
        for ch in children:
            t = ch.get_text(strip=True)[:25]
            if t:
                child_texts.append(f"{ch.name}:{t}")
        if child_texts:
            parts.append(f"children:[{', '.join(child_texts)}]")

        # DOM 형제 이웃 (최대 5개) — Dual-View +11.9%p 근사
        if parent and parent.name not in ("html", "body", "[document]", None):
            siblings = [s for s in parent.find_all(recursive=False)
                        if s != el and s.get_text(strip=True)][:5]
            nei_texts = [f"{s.name}:{s.get_text(strip=True)[:25]}" for s in siblings]
            if nei_texts:
                parts.append(f"near:[{', '.join(nei_texts)}]")

        return " | ".join(parts)
    except Exception:
        return ""


# ─────────────────────────────────────────────
# 계층적 A-Tree 포맷팅 (Hierarchy-aware)
# ─────────────────────────────────────────────

def _get_ancestor_path(soup, el) -> list:
    """요소의 조상 노드 리스트를 반환한다 (Root -> Parent)."""
    if not el:
        return []
    path = []
    curr = el.parent
    while curr and curr.name not in ("html", "body", "[document]", None):
        path.append(curr)
        curr = curr.parent
    return path[::-1]


def _format_node_minimal(el) -> str:
    """조상 노드를 위한 최소 정보 포맷 (tag, id, class, role)."""
    tag = el.name
    attrs = el.attrs
    parts = [f"<{tag}"]
    if "id" in attrs:
        parts.append(f"id={attrs['id']}")
    if "class" in attrs:
        cls = " ".join(attrs["class"]) if isinstance(attrs["class"], list) else str(attrs["class"])
        parts.append(f"class={cls[:30]}")
    role = attrs.get("role") or attrs.get("aria-label")
    if role:
        parts.append(f"role={str(role)[:30]}")
    return " ".join(parts) + ">"


def format_as_tree(candidates, soup=None, compact: bool = False) -> str:
    """후보 요소와 그 조상들을 포함하는 인덴트 기반 요약 트리를 생성한다.

    1. 모든 후보의 조상들을 '필수 노드'로 식별.
    2. DFS 탐색을 통해 필수 노드만 출력.
    3. 후보 노드는 번호[1]와 전체 속성을, 조상 노드는 최소 정보만 표시.
    """
    if soup is None:
        return format_numbered_candidates(candidates, soup=None, compact=compact)

    # 1. 필수 노드 매핑 (candidate -> soup_element)
    cand_to_el = {}
    relevant_nodes = set()
    for i, c in enumerate(candidates, 1):
        el = _find_element_in_soup(soup, c)
        if el:
            cand_to_el[i] = el
            relevant_nodes.add(el)
            for anc in _get_ancestor_path(soup, el):
                relevant_nodes.add(anc)

    if not cand_to_el:
        return format_numbered_candidates(candidates, soup=None, compact=compact)

    # 2. DFS 트리 생성
    _KEY_ATTRS_COMPACT = ("type", "name", "aria-label", "placeholder", "label", "options", "role")
    _KEY_ATTRS = ("id", "name", "type", "role", "aria-label", "placeholder", "value", "href", "for", "label", "options")

    lines = []
    
    def _render(node, depth):
        is_candidate = False
        cand_idx = None
        for idx, el in cand_to_el.items():
            if el == node:
                is_candidate = True
                cand_idx = idx
                break
        
        indent = "  " * depth
        if is_candidate:
            c = candidates[cand_idx - 1]
            tag = c.get("tag", "")
            text = str(c.get("text", "")).strip()
            attrs_dict = parse_attrs_str(str(c.get("attrs", "")))
            
            # SeeAct fallback text
            if not text:
                for fb in ("aria-label", "placeholder", "title", "role", "name"):
                    if attrs_dict.get(fb):
                        text = f"[{fb}:{attrs_dict[fb][:30]}]"
                        break
            
            key_list = _KEY_ATTRS_COMPACT if compact else _KEY_ATTRS
            attr_parts = [f"{k}={str(attrs_dict[k])[:50]}" for k in key_list if attrs_dict.get(k)]
            
            line = f"{indent}[{cand_idx}] <{tag}>"
            if text: line += f" '{text[:50]}'"
            if attr_parts: line += " | " + " | ".join(attr_parts)
            lines.append(line)
        else:
            lines.append(f"{indent}{_format_node_minimal(node)}")

        # 자식 노드 중 relevant_nodes에 속하는 것만 방문
        for child in node.find_all(recursive=False):
            if child in relevant_nodes:
                _render(child, depth + 1)

    # Root 영역들 찾기 (가장 상위의 relevant_nodes)
    roots = []
    for node in relevant_nodes:
        if not any(p in relevant_nodes for p in node.parents):
            roots.append(node)
    
    for r in roots:
        _render(r, 0)
        
    return "\n".join(lines)


def format_numbered_candidates(candidates, soup=None, compact: bool = False) -> str:
    """후보를 1~N 번호로 포맷팅한다. soup가 있으면 계층적 트리 모드로 동작."""
    if soup is not None:
        try:
            tree = format_as_tree(candidates, soup, compact)
            if tree.strip():
                return tree
        except Exception:
            pass # 실패 시 리스트 모드로 fallback

    # --- 기존 리스트 모드 (List Mode) ---
    _KEY_ATTRS = ("id", "name", "type", "role", "aria-label", "aria-labelledby",
                  "aria-describedby", "aria-expanded", "aria-selected", "aria-checked",
                  "placeholder", "value", "href", "for",
                  "required", "disabled", "label", "options")
    _KEY_ATTRS_COMPACT = ("type", "name", "aria-label", "placeholder", "label", "options", "role")
    _NOISE_PREFIXES = ("class", "style", "data-react", "data-v-", "data-test",
                       "xpath", "jsname", "jsaction", "jscontroller", "data-analytics",
                       "tabindex", "data_", "data-ga", "data-track")

    lines = []
    for i, c in enumerate(candidates, 1):
        tag  = c.get("tag", "")
        text = str(c.get("text", "")).strip()
        attrs_dict = parse_attrs_str(str(c.get("attrs", "")))

        if not text:
            for _fb_key in ("aria-label", "placeholder", "title", "role", "name"):
                _fb_val = attrs_dict.get(_fb_key, "")
                if _fb_val:
                    text = f"[{_fb_key}:{_fb_val[:40]}]"
                    break

        key_parts = []
        key_list = _KEY_ATTRS_COMPACT if compact else _KEY_ATTRS
        for k in key_list:
            v = attrs_dict.get(k, "")
            if v:
                key_parts.append(f"{k}={v[:60]}")

        if not compact:
            extras = []
            for k, v in attrs_dict.items():
                if k in _KEY_ATTRS or any(k.startswith(p) for p in _NOISE_PREFIXES):
                    continue
                extras.append(f"{k}={str(v)[:40]}")
                if len(extras) >= 2: break
            key_parts.extend(extras)

        line = f"{i:>2}. [{tag}]"
        if text: line += f' "{text[:60]}"'
        if key_parts: line += " | " + " | ".join(key_parts)
        lines.append(line)
    return "\n".join(lines)


def choice_to_candidate_id(choice, candidates) -> str:
    """번호(1~N)를 실제 candidate_id로 변환한다."""
    try:
        idx = int(choice) - 1
        if 0 <= idx < len(candidates):
            return str(candidates[idx].get("candidate_id", ""))
    except (ValueError, TypeError):
        pass
    return ""


# ─────────────────────────────────────────────
# 임베딩 기반 후보 재정렬 (real_web 전용)
# ─────────────────────────────────────────────

def _candidate_description(c) -> str:
    """후보 요소를 짧은 자연어로 변환한다. 있는 데이터만 사용 (할루시네이션 없음)."""
    tag   = c.get("tag", "")
    text  = str(c.get("text", "")).strip()
    attrs = parse_attrs_str(str(c.get("attrs", "")))
    label = (text
             or attrs.get("aria-label", "")
             or attrs.get("placeholder", "")
             or attrs.get("label", "")
             or attrs.get("name", ""))
    options = attrs.get("options", "")
    desc = f"{tag} {label}".strip()
    if options:
        desc += f" choices: {options[:60]}"
    return desc


_rerank_model = None


def rerank_candidates_by_embedding(task: str, candidates: list, use_rerank: bool = True) -> list:
    """task와 각 후보를 cross-encoder로 공동 인코딩해 내림차순 정렬한다 (전체 후보 정렬)."""
    if not use_rerank or not candidates:
        return candidates
    global _rerank_model
    if _rerank_model is None:
        from sentence_transformers import CrossEncoder
        _rerank_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
    descs  = [_candidate_description(c) for c in candidates]
    scores = _rerank_model.predict([(task, d) for d in descs])
    ranked = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    return [candidates[i] for i in ranked]


def hard_negative_shuffle(candidates: list, target_id: str) -> list:
    """셔플 후 정답과 같은 tag의 후보(hard negative)를 정답 바로 앞에 배치한다.

    같은 tag 후보가 없으면 일반 random.shuffle과 동일하게 동작한다.
    """
    shuffled = candidates.copy()
    random.shuffle(shuffled)

    target_pos = next(
        (i for i, c in enumerate(shuffled) if str(c.get("candidate_id", "")) == target_id),
        None,
    )
    if target_pos is None:
        return shuffled

    target_tag = shuffled[target_pos].get("tag", "")
    hn_pos = next(
        (i for i, c in enumerate(shuffled)
         if i != target_pos and c.get("tag", "") == target_tag),
        None,
    )
    # hard negative를 정답 바로 앞으로 swap (정답이 맨 앞이면 뒤로)
    if hn_pos is not None:
        swap_to = (target_pos - 1) if target_pos > 0 else (target_pos + 1)
        if swap_to < len(shuffled):
            shuffled[hn_pos], shuffled[swap_to] = shuffled[swap_to], shuffled[hn_pos]

    return shuffled


def generate_cot_reasoning(task: str, op: str, choice: int, candidate: dict, value: str,
                           candidates: list = None, soup=None) -> str:
    """학습 데이터용: task-aware + 구조 기반 추론 문장 생성.

    A-Tree 구조 정보를 반영하여 '어떤 섹션(parent)에 있는지'를 CoT에 포함하고,
    유사한 오답(sibling)과의 차별점을 강조하여 grounding 능력을 키운다.
    """
    tag   = candidate.get("tag", "")
    text  = str(candidate.get("text", "")).strip()
    attrs = parse_attrs_str(str(candidate.get("attrs", "")))
    label = (text or attrs.get("aria-label", "") or attrs.get("placeholder", "") or 
             attrs.get("label", "") or attrs.get("name", "")).strip()

    # 1. 구조적 경로 파악 (Hierarchy path)
    ctx_path = []
    if soup:
        el = _find_element_in_soup(soup, candidate)
        if el:
            curr = el.parent
            while curr and curr.name not in ("html", "body", "[document]", None):
                p_tag = curr.name
                # id, aria-label, role 순으로 힌트 추출
                p_hint = curr.get("id") or curr.get("aria-label") or curr.get("role") or ""
                ctx_path.append(f"{p_tag}{'#' + str(p_hint)[:15] if p_hint else ''}")
                curr = curr.parent
                if len(ctx_path) >= 2: break # 상위 2단계까지만 포함하여 간결성 유지
    
    path_str = " > ".join(reversed(ctx_path))
    ctx_hint = f" located under {path_str}," if path_str else ""

    task_clean = str(task).strip()[:40]
    el_desc = f"element [{choice}] <{tag}> '{label[:20]}'"

    # 2. 비교 추론 (Sibling/Similar tag 차별화)
    contrast = ""
    if candidates:
        similars = [i for i, c in enumerate(candidates, 1) 
                    if i != choice and str(c.get("tag", "")) == tag][:1]
        if similars:
            contrast = f" (choosing this over El.{similars[0]} due to closer task relevance)"

    if op == "CLICK":
        reason = f"To '{task_clean}',{ctx_hint} I need to CLICK {el_desc}{contrast}."
    elif op == "TYPE":
        reason = f"For '{task_clean}',{ctx_hint} I will TYPE '{value[:20]}' into {el_desc}{contrast}."
    else:  # SELECT
        reason = f"To fulfill '{task_clean}',{ctx_hint} I must SELECT '{value[:20]}' using {el_desc}{contrast}."

    return reason[:200]


# ─────────────────────────────────────────────
# History 압축 (AgentOccam: 피봇 step만 남김)
# ─────────────────────────────────────────────

def _compress_history(history_str: str) -> str:
    """AgentOccam: tag+text+op만 남겨 60-70% 토큰 절약.

    입력: "Step 1: [button] Submit -> CLICK\nStep 2: [input] Email -> TYPE: foo@bar.com"
    출력: "[button]Submit→CLICK | [input]Email→TYPE:foo@bar.com"
    """
    if not history_str or str(history_str).strip() in ("", "None", "nan"):
        return "None"
    steps = re.findall(
        r'Step\s+\d+:\s*\[([^\]]+)\]\s*(.*?)\s*->\s*(\w+)(?::\s*([^\n]*))?',
        str(history_str)
    )
    if not steps:
        return str(history_str)[:300]
    compressed = []
    for tag, text, op, value in steps:
        text = text.strip()[:30]
        entry = f"[{tag}]{' ' + text if text else ''}→{op}"
        if value and op != "CLICK":
            entry += f":{value.strip()[:20]}"
        compressed.append(entry)
    return " | ".join(compressed)


# ─────────────────────────────────────────────
# LCA (Lowest Common Ancestor) 분석 (DPO용)
# ─────────────────────────────────────────────

def get_dom_distance(soup, el1, el2) -> int:
    """두 BeautifulSoup 요소 간의 DOM 트리 거리를 계산한다.
    거리 = depth(el1) + depth(el2) - 2 * depth(LCA(el1, el2))
    """
    if not el1 or not el2:
        return 999  # 거리를 측정할 수 없는 경우
    if el1 == el2:
        return 0

    # 각 요소의 조상 노드 리스트 (root 방향)
    path1 = [el1] + list(el1.parents)
    path2 = [el2] + list(el2.parents)

    # Root부터 내려오면서 공통 조상이 끝나는 지점 찾기
    path1.reverse()
    path2.reverse()

    common_depth = 0
    for n1, n2 in zip(path1, path2):
        if n1 == n2:
            common_depth += 1
        else:
            break

    return (len(path1) + len(path2)) - (2 * common_depth)


def _is_false_negative(cand, target_cand):
    """정답(target_cand)과 지나치게 유사한 후보인지 판별한다. (False Negative 필터)"""
    if not cand or not target_cand:
        return True
    if str(cand.get("candidate_id")) == str(target_cand.get("candidate_id")):
        return True

    t1 = str(cand.get("text", "")).strip().lower()
    t2 = str(target_cand.get("text", "")).strip().lower()

    # 1. 강한 제외 조건: text가 비어있지 않고 완전히 동일
    if t1 and t1 == t2:
        return True

    a1 = parse_attrs_str(str(cand.get("attrs", "")))
    a2 = parse_attrs_str(str(target_cand.get("attrs", "")))

    # 2. 강한 제외 조건: label, aria-label 중 하나라도 비어있지 않고 완전히 동일
    for key in ["label", "aria-label"]:
        v1 = str(a1.get(key, "")).strip().lower()
        v2 = str(a2.get(key, "")).strip().lower()
        if v1 and v1 == v2:
            return True

    # 3. 약한 유사성 조건: name, placeholder 일치 여부 확인
    weak_matches = 0
    for key in ["name", "placeholder"]:
        v1 = str(a1.get(key, "")).strip().lower()
        v2 = str(a2.get(key, "")).strip().lower()
        if v1 and v1 == v2:
            weak_matches += 1

    # 4. 약한 조건이 2개 이상 일치하면 제외
    if weak_matches >= 2:
        return True

    return False


def find_lca_hard_negative(row, candidates, target_id):
    """15개 후보 중 정답(target_id)과 가장 헷갈릴만한 '매력적인 오답'을 반환한다.
    
    WEPO+ 파이프라인:
    1. DOM 거리(LCA)가 가까움 (구조적 유사도)
    2. Reranker 점수가 높음 (의미적 유사도)
    3. False-negative 필터 적용
    """
    from bs4 import BeautifulSoup

    # 1. 정답 요소 찾기 및 기본 필터링
    target_cand = next((c for c in candidates if str(c.get("candidate_id", "")) == target_id), None)
    others = [c for c in candidates if str(c.get("candidate_id", "")) != target_id]
    
    if not target_cand or not others:
        return random.choice(others) if others else None

    # False-negative 필터링 (정답과 너무 비슷한 건 오답 샘플로 부적합)
    filtered_others = [c for c in others if not _is_false_negative(c, target_cand)]
    if not filtered_others:
        filtered_others = others

    html_str = str(row.get("cleaned_html", ""))
    soup = None
    if html_str and html_str != "nan":
        try:
            soup = BeautifulSoup(html_str, "lxml")
        except: pass

    target_el = _find_element_in_soup(soup, target_cand) if soup else None

    # 2. 하이브리드 스코어링 (LCA + Reranker)
    # Reranker 점수를 미리 계산하기 위해 candidates 리스트 활용
    task = str(row.get("task", ""))
    reranked_others = rerank_candidates_by_embedding(task, filtered_others)
    
    # Rerank 순위 기반 점수 (앞쪽에 올수록 높은 점수)
    rerank_scores = {c["candidate_id"]: (len(reranked_others) - i) / len(reranked_others) 
                     for i, c in enumerate(reranked_others)}

    best_neg = None
    max_total_score = -999.0

    for c in filtered_others:
        # A. 물리적 점수 (LCA)
        dist_score = 0.0
        sibling_bonus = 0.0
        if soup and target_el:
            c_el = _find_element_in_soup(soup, c)
            if c_el:
                dist = get_dom_distance(soup, target_el, c_el)
                dist_score = max(0, 20 - dist) / 20.0
                if c_el.parent == target_el.parent:
                    sibling_bonus = 0.5 # 같은 부모면 매우 강력한 오답 후보

        # B. 의미적 점수 (Reranker)
        semantic_score = rerank_scores.get(c["candidate_id"], 0.0)
        
        # C. 태그 보너스
        tag_bonus = 0.2 if str(c.get("tag")).lower() == str(target_cand.get("tag")).lower() else 0.0

        total_score = dist_score + sibling_bonus + (semantic_score * 1.5) + tag_bonus

        if total_score > max_total_score:
            max_total_score = total_score
            best_neg = c

    return best_neg or random.choice(filtered_others)


# ─────────────────────────────────────────────
# RAG 포맷팅 & 프롬프트 빌더
# ─────────────────────────────────────────────
RETRIEVAL_K             = 3
RETRIEVAL_TASK_CHARS    = 150
RETRIEVAL_HISTORY_CHARS = 150


def format_similar_examples(examples, max_task_chars=None, max_history_chars=None):
    if not examples:
        return "[Similar Past Examples]\nNone"

    def _trunc(s, n):
        if n is None or s is None:
            return s
        s = str(s)
        return s if len(s) <= n else s[:n].rstrip() + "..."

    lines = ["[Similar Past Examples]"]
    for i, ex in enumerate(examples, 1):
        lines.extend([
            f"Example {i}:",
            f"  Task: {_trunc(ex.get('task', ''), max_task_chars)}",
            f"  History: {_trunc(ex.get('history', ''), max_history_chars)}",
            f"  Action: op={ex.get('target_op', '')}, label=\"{ex.get('target_label', '')}\", value=\"{ex.get('target_value', '')}\"",
        ])
    return "\n".join(lines)


def build_prompt(
    row,
    candidates,
    retriever=None,
    k=RETRIEVAL_K,
    exclude_id=None,
    compact_candidates: bool = False,
    use_rerank: bool = True,
):
    """Optimized English-only prompt for Qwen3-32B Agentic Reasoning."""
    from bs4 import BeautifulSoup

    html_type   = detect_html_type(row)
    html_str    = str(row.get("cleaned_html", ""))
    task_str    = str(row.get("task", ""))
    history_str = _compress_history(str(row.get("history", "")))

    # 1. Universal Rerank (Put the most relevant elements at the top)
    candidates = rerank_candidates_by_embedding(task_str, candidates, use_rerank=use_rerank)
    n = len(candidates)

    # 2. RAG (Inject past successful examples)
    rag_block = ""
    if retriever:
        examples = retriever.query(row, k=k, exclude_id=exclude_id)
        rag_block = format_similar_examples(examples, RETRIEVAL_TASK_CHARS, RETRIEVAL_HISTORY_CHARS) + "\n\n"

    # 3. HTML Parsing & A-Tree formatting
    soup = None
    if html_str and html_str != "nan":
        try:
            soup = BeautifulSoup(html_str, "lxml")
        except: soup = None
    numbered = format_numbered_candidates(candidates, soup=soup, compact=compact_candidates)

    # 4. Thinking Guidelines (Force logical grounding in English)
    reasoning_guideline = """[Reasoning Guidelines]
1. Analyze the 'Task' and 'History' to identify the current objective.
2. Scan the 'Candidate Elements' (A-Tree) for labels, IDs, or ARIA attributes that match the task requirements.
3. Reference 'Similar Past Examples' to see how similar tasks were successfully handled.
4. Distinguish between similar elements by checking their structural paths (ancestors) and surrounding neighbors.
5. Select the single best element and determine the correct operation (CLICK, TYPE, or SELECT)."""

    # 5. Core Instruction Assembly
    role_instruction = "You are a specialized Web UI automation agent. Your goal is to predict the next correct action based on the current state of the web page."
    
    prompt = f"""{role_instruction}

{rag_block}[Candidate Elements] (Ranked by relevance, 1-{n})
{numbered}

[Task]
{task_str}

[History]
{history_str}

{reasoning_guideline}

[Output Format]
Provide your internal reasoning process in English inside <think>...</think> tags.
Then, output the final action in the following JSON format:
{{"op": "CLICK|TYPE|SELECT", "choice": <number 1-{n}>, "value": "string or empty"}}"""

    # Add workflow-specific status if applicable
    if html_type == "workflow":
        ctx = extract_workflow_context(html_str)
        workflow_hint = f"\n\n[Workflow Status]\nProgress: step {ctx['current_step']} of {ctx['total_steps']}\n"
        if ctx["completed_fields"]:
            workflow_hint += f"Completed: {', '.join(ctx['completed_fields'])} (Do NOT interact with these)\n"
        return prompt + workflow_hint
    
    return prompt
'''

with open('/content/src/preprocess.py', 'w', encoding='utf-8') as f:
    f.write(preprocess_py)

train_py = r'''# -*- coding: utf-8 -*-
import os
import argparse
import torch
import pandas as pd
import json
import re
import random
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig, DPOTrainer, DPOConfig
from datasets import Dataset
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.metrics import confusion_matrix

from preprocess import (
    fallback_rule_based,
    extract_value_from_task,
    enforce_consistency,
    detect_html_type,
    extract_workflow_context,
    format_numbered_candidates,
    choice_to_candidate_id,
    build_prompt,
    RETRIEVAL_K,
    generate_cot_reasoning,
    hard_negative_shuffle,
    find_lca_hard_negative,
)
from retrieval import ExampleRetriever


# ─────────────────────────────────────────────
# 런타임 플래그 (변경해야 할 설정은 여기서만)
# ─────────────────────────────────────────────
MAX_SEQ_LENGTH            = 4096
EVAL_SAMPLE_SIZE          = 500   # 80GB에서는 더 넓은 샘플로 검증 가능
USE_RETRIEVAL             = True  # RAG 복구
USE_CONSISTENCY           = True
VALIDATION_MODE           = True  # False 로 바꾸면 전체 데이터 학습
FINAL_TRAIN_ON_FULL_DATA  = False # True 로 바꾸면 val 없이 전체 학습
USE_DPO                   = True  # SFT 대신 DPO 사용 여부

# 학습 제외할 site_token (CLICK 편향 + test에 없음 — 팀원 분석)
EXCLUDE_SITES = {"site_2aa627db"}

# Augmentation & OOF
SHUFFLE_AUGMENT_N    = 1          # 원본 1 + 셔플 N = (1+N)배 데이터 (real_web은 2배 적용)
OOF_N_FOLDS          = 3          # OOF fold 수

# ── Qwen3-32B Unsloth 4bit + 단일 A100 80GB (최신 Qwen3 적용) ──
BASE_MODEL_ID = "unsloth/Qwen3-32B-unsloth-bnb-4bit"
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]
LORA_R = 16
LORA_ALPHA = 32

# DPO는 (chosen+rejected)로 메모리 부담이 크지만, 32B 4bit + 80GB면 아래가 일반적으로 여유 있음.
# OOM 시 DPO_PER_DEVICE_BATCH만 8 -> 4 순으로 낮출 것.
DPO_PER_DEVICE_BATCH   = 32
DPO_GRAD_ACCUM_STEPS   = 1

SFT_PER_DEVICE_BATCH   = 64
SFT_GRAD_ACCUM_STEPS   = 1

DATALOADER_NUM_WORKERS = 8
SFT_DATASET_NUM_PROC   = 8


def configure_a100_throughput() -> None:
    """TF32로 matmul·conv 가속 (A100)."""
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True


def load_base_model_and_tokenizer(max_seq_length: int):
    """bnb 4bit 단일 GPU 적재. device_map 고정으로 CPU 오프로드·bnb 검증 오류 방지."""
    return FastLanguageModel.from_pretrained(
        model_name     = BASE_MODEL_ID,
        max_seq_length = max_seq_length,
        load_in_4bit   = True,
        device_map     = {"": 0},
    )


def prepare_tokenizer(tokenizer) -> None:
    tokenizer.padding_side = "left"
    tokenizer.truncation_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token


def attach_lora(model, random_state: int):
    return FastLanguageModel.get_peft_model(
        model,
        r               = LORA_R,
        target_modules  = LORA_TARGET_MODULES,
        lora_alpha      = LORA_ALPHA,
        lora_dropout    = 0,
        bias            = "none",
        use_gradient_checkpointing = "unsloth",
        random_state    = random_state,
    )


def _dpo_training_kwargs():
    return dict(
        per_device_train_batch_size = DPO_PER_DEVICE_BATCH,
        gradient_accumulation_steps = DPO_GRAD_ACCUM_STEPS,
        dataloader_num_workers      = DATALOADER_NUM_WORKERS,
        dataloader_pin_memory     = True,
        dataloader_persistent_workers = DATALOADER_NUM_WORKERS > 0,
    )


def _sft_training_kwargs():
    return dict(
        per_device_train_batch_size = SFT_PER_DEVICE_BATCH,
        gradient_accumulation_steps = SFT_GRAD_ACCUM_STEPS,
        dataloader_num_workers      = DATALOADER_NUM_WORKERS,
        dataloader_pin_memory     = True,
        dataloader_persistent_workers = DATALOADER_NUM_WORKERS > 0,
    )


# ─────────────────────────────────────────────
# JSON 파싱
# ─────────────────────────────────────────────

def _parse_json_safe(text: str) -> dict | None:
    """비중첩 JSON 블록을 역순으로 파싱 시도 (thinking 내 중괄호 오파싱 방지)."""
    for m in reversed(list(re.finditer(r'\{[^{}]*\}', text, re.DOTALL))):
        try:
            return json.loads(m.group(0))
        except json.JSONDecodeError:
            continue
    return None


def extract_json_answer(response, candidates):
    """LLM 응답에서 JSON을 파싱하고 choice 번호를 candidate_id로 변환한다."""
    try:
        search_text = response.split('</think>', 1)[-1] if '</think>' in response else response
        data = _parse_json_safe(search_text)
        if data is None:
            return {"op": "CLICK", "target_id": "", "value": ""}
        op = str(data.get("op", "CLICK")).upper()
        if op not in ("CLICK", "TYPE", "SELECT"):
            op = "CLICK"
        value     = "" if op == "CLICK" else str(data.get("value", ""))
        choice    = data.get("choice", "")
        target_id = choice_to_candidate_id(choice, candidates)
        return {"op": op, "target_id": target_id, "value": value}
    except Exception:
        return {"op": "CLICK", "target_id": "", "value": ""}


# ─────────────────────────────────────────────
# 학습 데이터 생성
# ─────────────────────────────────────────────

def prepare_training_data(
    df,
    retriever=None,
    shuffle_n=SHUFFLE_AUGMENT_N,
    realweb_aug_boost: int = 1,
    compact_prompt: bool = False,
):
    """DPO 또는 SFT용 학습 데이터를 생성한다.

    WEPO 프레임워크 적용 (DPO Mode):
    1. Chosen: 정답 요소 (aw)
    2. Rejected: LCA 거리가 가장 가까운 오답 요소 (al).
       - 변별력 극대화를 위해 op/value는 chosen과 동일하게 유지 (요소 선택에만 집중).
    3. f_op 휴리스틱: 정답이 TYPE/SELECT면 오답을 33% 확률로 CLICK으로 변조하여 기능적 변별력 추가.
    """
    mode_str = "DPO" if USE_DPO else "SFT"
    print(
        f"Preparing training data ({mode_str} Mode, shuffle_n={shuffle_n}, "
        f"realweb_aug_boost={realweb_aug_boost}, compact_prompt={compact_prompt})..."
    )
    instructions = []
    skipped = 0

    for _, row in df.iterrows():
        try:
            candidates = json.loads(row["candidate_elements"])
        except Exception:
            skipped += 1
            continue

        target_id = str(row.get("target_id", ""))
        chosen_choice = None
        for i, c in enumerate(candidates, 1):
            if str(c.get("candidate_id", "")) == target_id:
                chosen_choice = i
                break

        if chosen_choice is None:
            skipped += 1
            continue

        chosen_val = str(row["value"]) if pd.notna(row.get("value")) else ""
        if row["op"] == "CLICK":
            chosen_val = ""

        # HTML 파싱 (A-Tree 및 CoT 용)
        from bs4 import BeautifulSoup
        html_str = str(row.get("cleaned_html", ""))
        soup = None
        if html_str and html_str != "nan":
            try:
                soup = BeautifulSoup(html_str, "lxml")
            except Exception:
                soup = None

        # ── 공통 Prompt ──
        prompt = build_prompt(
            row, candidates,
            retriever=retriever,
            k=RETRIEVAL_K,
            exclude_id=row.get("id"),
            compact_candidates=compact_prompt,
            use_rerank=False,
        )

        # ── 1. Chosen Action ──
        target_cand = candidates[chosen_choice - 1]
        chosen_reasoning = generate_cot_reasoning(
            str(row.get("task", "")), row["op"], chosen_choice, target_cand, chosen_val, 
            candidates=candidates, soup=soup
        )
        chosen_output = f"<think>\n{chosen_reasoning}\n</think>\n" + json.dumps(
            {"op": row["op"], "choice": chosen_choice, "value": chosen_val}
        )

        if USE_DPO:
            # ── 2. Rejected Action (LCA 기반) ──
            rejected_cand = find_lca_hard_negative(row, candidates, target_id)
            if not rejected_cand:
                skipped += 1
                continue

            rejected_id = str(rejected_cand.get("candidate_id", ""))
            rejected_choice = next(i for i, c in enumerate(candidates, 1) if str(c.get("candidate_id", "")) == rejected_id)

            # f_op 휴리스틱: TYPE/SELECT 시 33% 확률로 CLICK 변조
            rejected_op = row["op"]
            rejected_val = chosen_val
            if row["op"] in ("TYPE", "SELECT") and random.random() < 0.33:
                rejected_op = "CLICK"
                rejected_val = ""

            rejected_reasoning = generate_cot_reasoning(
                str(row.get("task", "")), rejected_op, rejected_choice, rejected_cand, rejected_val, 
                candidates=candidates, soup=soup
            )
            rejected_output = f"<think>\n{rejected_reasoning}\n</think>\n" + json.dumps(
                {"op": rejected_op, "choice": rejected_choice, "value": rejected_val}
            )

            instructions.append({
                "prompt":   f"### Instruction:\n{prompt}\n\n### Response:\n",
                "chosen":   chosen_output,
                "rejected": rejected_output
            })
        else:
            # SFT 모드
            instructions.append({
                "instruction": prompt,
                "input": "",
                "output": chosen_output
            })

        # ── 셔플 Augmentation (과적합 방지 핵심) ──
        # real_web은 구조 암기 방지를 위해 더 많은 셔플 적용
        realweb_mult = 3 if detect_html_type(row) == "real_web" else 1
        effective_shuffle = shuffle_n * realweb_mult
        
        for _ in range(effective_shuffle):
            shuffled = hard_negative_shuffle(candidates, target_id)
            new_chosen_choice = next(i for i, c in enumerate(shuffled, 1) if str(c.get("candidate_id", "")) == target_id)

            aug_prompt = build_prompt(
                row, shuffled,
                retriever=retriever,
                k=RETRIEVAL_K,
                exclude_id=row.get("id"),
                compact_candidates=compact_prompt,
                use_rerank=False,
            )

            aug_chosen_cand = shuffled[new_chosen_choice - 1]
            aug_chosen_reasoning = generate_cot_reasoning(
                str(row.get("task", "")), row["op"], new_chosen_choice, aug_chosen_cand, chosen_val, 
                candidates=shuffled, soup=soup
            )
            aug_chosen_output = f"<think>\n{aug_chosen_reasoning}\n</think>\n" + json.dumps(
                {"op": row["op"], "choice": new_chosen_choice, "value": chosen_val}
            )

            if USE_DPO:
                new_rejected_choice = next(i for i, c in enumerate(shuffled, 1) if str(c.get("candidate_id", "")) == rejected_id)
                aug_rejected_cand = shuffled[new_rejected_choice - 1]
                aug_rejected_reasoning = generate_cot_reasoning(
                    str(row.get("task", "")), rejected_op, new_rejected_choice, aug_rejected_cand, rejected_val, 
                    candidates=shuffled, soup=soup
                )
                aug_rejected_output = f"<think>\n{aug_rejected_reasoning}\n</think>\n" + json.dumps(
                    {"op": rejected_op, "choice": new_rejected_choice, "value": rejected_val}
                )
                instructions.append({
                    "prompt":   f"### Instruction:\n{aug_prompt}\n\n### Response:\n",
                    "chosen":   aug_chosen_output,
                    "rejected": aug_rejected_output
                })
            else:
                instructions.append({
                    "instruction": aug_prompt,
                    "input": "",
                    "output": aug_chosen_output
                })

    print(f"  총 {len(instructions)}개 예시 생성 ({mode_str} 모드, skip: {skipped}개)")
    return instructions


# ─────────────────────────────────────────────
# 검증
# ─────────────────────────────────────────────

def evaluate_full_pipeline(
    model,
    tokenizer,
    val_df,
    retriever,
    train_sites,
    base_dir,
    max_seq_length=MAX_SEQ_LENGTH,
    compact_prompt: bool = False,
):
    print("\n🧪 Running held-out evaluation...")
    FastLanguageModel.for_inference(model)
    tokenizer.padding_side  = "left"
    tokenizer.truncation_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    total_n = len(val_df)
    if EVAL_SAMPLE_SIZE is not None and total_n > EVAL_SAMPLE_SIZE:
        val_df = val_df.sample(n=EVAL_SAMPLE_SIZE, random_state=42).reset_index(drop=True)
        print(f"Sampled eval: {len(val_df)}/{total_n} rows")

    rows = []
    y_true_ops, y_pred_ops = [], []

    for _, row in val_df.iterrows():
        try:
            candidates = json.loads(row["candidate_elements"])
        except Exception:
            candidates = []

        prompt  = build_prompt(
            row,
            candidates,
            retriever=retriever,
            k=RETRIEVAL_K,
            exclude_id=row.get("id"),
            compact_candidates=compact_prompt,
        )
        text    = f"### Instruction:\n{prompt}\n\n### Response:\n<think>\n"
        inputs  = tokenizer([text], return_tensors="pt", padding=True, truncation=True,
                            max_length=max_seq_length).to("cuda")
        outputs = model.generate(**inputs, max_new_tokens=512, use_cache=True,
                                 do_sample=False, pad_token_id=tokenizer.eos_token_id)
        generated = outputs[:, inputs["input_ids"].shape[1]:]
        response  = tokenizer.batch_decode(generated, skip_special_tokens=True)[0]
        pred      = extract_json_answer(response, candidates)

        # fallback: choice 변환 실패 시
        valid_ids = {str(c.get("candidate_id", "")) for c in candidates}
        if str(pred["target_id"]) not in valid_ids:
            pred = fallback_rule_based(row, candidates)
        else:
            matched = next((c for c in candidates
                            if str(c.get("candidate_id", "")) == str(pred["target_id"])), None)
            attrs = str(matched.get("attrs", "")) if matched else ""
            # LLM value 우선, 빈 문자열이면 rule 보조
            if not pred["value"] and pred["op"] != "CLICK":
                pred["value"] = extract_value_from_task(row["task"], pred["op"], attrs)

        pred["_task"] = row["task"]
        if USE_CONSISTENCY:
            pred = enforce_consistency(pred, candidates)

        true_value = "" if pd.isna(row.get("value", "")) else str(row.get("value", ""))
        if row["op"] == "CLICK":
            true_value = ""

        item = {
            "id": row.get("id", ""),
            "site_seen": str(row.get("site_token", "")) in train_sites,
            "html_type": detect_html_type(row),
            "true_op":        str(row["op"]),
            "pred_op":        str(pred["op"]),
            "true_target_id": str(row["target_id"]),
            "pred_target_id": str(pred["target_id"]),
            "true_value":     true_value,
            "pred_value":     str(pred["value"]),
        }
        item["op_correct"]     = item["true_op"]        == item["pred_op"]
        item["target_correct"] = item["true_target_id"] == item["pred_target_id"]
        item["value_correct"]  = item["true_value"]     == item["pred_value"]
        item["exact_correct"]  = item["op_correct"] and item["target_correct"] and item["value_correct"]
        rows.append(item)
        y_true_ops.append(item["true_op"])
        y_pred_ops.append(item["pred_op"])

    eval_df = pd.DataFrame(rows)

    def summarize(frame):
        if len(frame) == 0:
            return {"n": 0, "op_acc": None, "target_id_acc": None,
                    "value_acc": None, "exact_match": None}
        return {
            "n":            int(len(frame)),
            "op_acc":       float(frame["op_correct"].mean()),
            "target_id_acc": float(frame["target_correct"].mean()),
            "value_acc":    float(frame["value_correct"].mean()),
            "exact_match":  float(frame["exact_correct"].mean()),
        }

    metrics = {
        "overall":      summarize(eval_df),
        "site_seen":    summarize(eval_df[eval_df["site_seen"]]),
        "site_unseen":  summarize(eval_df[~eval_df["site_seen"]]),
        "workflow":     summarize(eval_df[eval_df["html_type"] == "workflow"]),
        "real_web":     summarize(eval_df[eval_df["html_type"] == "real_web"]),
    }

    labels = ["CLICK", "TYPE", "SELECT"]
    cm = confusion_matrix(y_true_ops, y_pred_ops, labels=labels)
    print("\nConfusion matrix (rows=true, cols=pred):")
    print(pd.DataFrame(cm, index=labels, columns=labels).to_string())
    print(f"\nOverall: {metrics['overall']}")
    print(f"Workflow: {metrics['workflow']}")
    print(f"Real-web: {metrics['real_web']}")

    out_dir = os.path.join(base_dir, "outputs")
    os.makedirs(out_dir, exist_ok=True)
    with open(os.path.join(out_dir, "eval_metrics.json"), "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)
    return metrics


# ─────────────────────────────────────────────
# 학습 메인
# ─────────────────────────────────────────────

def train(e2_realweb_boost: bool = False, e3_compact_prompt: bool = False):
    assert torch.cuda.is_available(), (
        "CUDA GPU not available. GPU runtime (Colab T4/L4 or local CUDA) required."
    )
    gpu_name     = torch.cuda.get_device_name(0)
    total_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
    print(f"GPU: {gpu_name} | VRAM: {total_mem_gb:.1f} GB")

    configure_a100_throughput()
    max_seq_length = MAX_SEQ_LENGTH
    model, tokenizer = load_base_model_and_tokenizer(max_seq_length)
    prepare_tokenizer(tokenizer)
    model = attach_lora(model, random_state=3407)

    base_dir   = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
    train_path = os.path.join(base_dir, "data", "train.csv")
    df         = pd.read_csv(train_path)

    # site_2aa627db 제외 (CLICK 편향 96.5% + test에 없음)
    before = len(df)
    df = df[~df["site_token"].isin(EXCLUDE_SITES)].reset_index(drop=True)
    print(f"Excluded sites {EXCLUDE_SITES}: {before} -> {len(df)} rows")

    if VALIDATION_MODE and not FINAL_TRAIN_ON_FULL_DATA:
        splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
        train_idx, val_idx = next(splitter.split(df, groups=df["site_token"]))
        train_df = df.iloc[train_idx].reset_index(drop=True)
        val_df   = df.iloc[val_idx].reset_index(drop=True)
        print(f"Split: train={len(train_df)} rows / val={len(val_df)} rows "
              f"(train_sites={train_df['site_token'].nunique()}, "
              f"val_sites={val_df['site_token'].nunique()})")
    else:
        train_df = df.reset_index(drop=True)
        val_df   = None
        print(f"Final training mode: {len(train_df)} rows, "
              f"{train_df['site_token'].nunique()} sites")

    retriever  = ExampleRetriever().build(train_df) if USE_RETRIEVAL else None
    realweb_boost = 2 if e2_realweb_boost else 1
    train_data = prepare_training_data(
        train_df,
        retriever=retriever,
        realweb_aug_boost=realweb_boost,
        compact_prompt=e3_compact_prompt,
    )
    dataset    = Dataset.from_list(train_data)

    # ── Trainer 설정 ──
    if USE_DPO:
        print(f"\nStarting DPO training (Qwen3-32B Unsloth 4bit, A100 80GB, 1 epoch, beta=0.1)...")
        trainer = DPOTrainer(
            model            = model,
            ref_model        = None, # PEFT 시 None 이면 자동으로 생성
            processing_class = tokenizer,
            train_dataset    = dataset,
            args = DPOConfig(
                **_dpo_training_kwargs(),
                warmup_ratio               = 0.1,
                num_train_epochs           = 1,
                max_steps                  = -1,
                learning_rate              = 5e-5,
                fp16                       = False,
                bf16                       = True,
                logging_steps              = 10,
                save_strategy              = "epoch",
                save_total_limit           = 3,
                optim                      = "adamw_8bit",
                weight_decay               = 0.01,
                lr_scheduler_type          = "cosine",
                seed                       = 3407,
                output_dir                 = os.path.join(base_dir, "outputs"),
                beta                       = 0.1,
                max_prompt_length          = 2048,
                max_length                 = 4096,
            ),
        )
    else:
        print(f"\nStarting SFT training (Qwen3-32B Unsloth 4bit, A100 80GB, 1 epoch)...")
        # SFT 전용 포맷팅 (DPO는 필요 없음)
        def formatting_prompts_func(examples):
            texts = []
            for instruction, output in zip(examples["instruction"], examples["output"]):
                texts.append(f"### Instruction:\n{instruction}\n\n### Response:\n{output}")
            return {"text": texts}
        dataset = dataset.map(formatting_prompts_func, batched=True)

        trainer = SFTTrainer(
            model            = model,
            processing_class = tokenizer,
            train_dataset    = dataset,
            args = SFTConfig(
                **_sft_training_kwargs(),
                dataset_text_field         = "text",
                max_seq_length             = max_seq_length,
                dataset_num_proc           = SFT_DATASET_NUM_PROC,
                padding_free               = True,
                warmup_ratio               = 0.1,
                num_train_epochs           = 1,
                max_steps                  = -1,
                learning_rate              = 1e-4,
                fp16                       = False,
                bf16                       = True,
                logging_steps              = 20,
                save_strategy              = "epoch",
                save_total_limit           = 3,
                optim                      = "adamw_8bit",
                weight_decay               = 0.01,
                lr_scheduler_type          = "cosine",
                seed                       = 3407,
                output_dir                 = os.path.join(base_dir, "outputs"),
            ),
        )

    trainer.train()

    model.save_pretrained(os.path.join(base_dir, "lora_model"))
    tokenizer.save_pretrained(os.path.join(base_dir, "lora_model"))
    print("Training complete. Saved to 'lora_model'.")

    if VALIDATION_MODE and val_df is not None:
        train_sites = set(train_df["site_token"].astype(str))
        evaluate_full_pipeline(
            model,
            tokenizer,
            val_df,
            retriever,
            train_sites,
            base_dir,
            max_seq_length,
            compact_prompt=e3_compact_prompt,
        )
    else:
        print("Skipped held-out evaluation (final training mode).")


def train_oof(e2_realweb_boost: bool = False, e3_compact_prompt: bool = False):
    """3-Fold OOF 학습: fold별로 LoRA 학습 → 검증 → 어댑터 저장."""
    assert torch.cuda.is_available(), "CUDA GPU required."
    gpu_name = torch.cuda.get_device_name(0)
    total_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
    print(f"GPU: {gpu_name} | VRAM: {total_mem_gb:.1f} GB")
    print(f"OOF Mode: {OOF_N_FOLDS} folds, shuffle_n={SHUFFLE_AUGMENT_N}")
    configure_a100_throughput()

    max_seq_length = MAX_SEQ_LENGTH
    base_dir   = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
    train_path = os.path.join(base_dir, "data", "train.csv")
    df         = pd.read_csv(train_path)

    before = len(df)
    df = df[~df["site_token"].isin(EXCLUDE_SITES)].reset_index(drop=True)
    print(f"Excluded sites {EXCLUDE_SITES}: {before} -> {len(df)} rows")

    gkf = GroupKFold(n_splits=OOF_N_FOLDS)
    all_oof_metrics = []

    for fold, (train_idx, val_idx) in enumerate(gkf.split(df, groups=df["site_token"])):
        print(f"\n{'='*60}")
        print(f"  FOLD {fold}/{OOF_N_FOLDS - 1}")
        print(f"{'='*60}")

        train_fold = df.iloc[train_idx].reset_index(drop=True)
        val_fold   = df.iloc[val_idx].reset_index(drop=True)
        print(f"  train={len(train_fold)} ({train_fold['site_token'].nunique()} sites) / "
              f"val={len(val_fold)} ({val_fold['site_token'].nunique()} sites)")

        # 각 fold마다 모델을 새로 로드 (이전 fold 가중치 오염 방지)
        model, tokenizer = load_base_model_and_tokenizer(max_seq_length)
        prepare_tokenizer(tokenizer)
        model = attach_lora(model, random_state=3407 + fold)

        retriever  = ExampleRetriever().build(train_fold) if USE_RETRIEVAL else None
        realweb_boost = 2 if e2_realweb_boost else 1
        train_data = prepare_training_data(
            train_fold,
            retriever=retriever,
            realweb_aug_boost=realweb_boost,
            compact_prompt=e3_compact_prompt,
        )
        dataset    = Dataset.from_list(train_data)

        fold_output_dir = os.path.join(base_dir, f"outputs_fold_{fold}")

        if USE_DPO:
            print(f"\n  Starting fold {fold} DPO (Qwen3-32B 4bit, A100 80GB, 1 epoch, beta=0.1)...")
            trainer = DPOTrainer(
                model            = model,
                ref_model        = None,
                processing_class = tokenizer,
                train_dataset    = dataset,
                args = DPOConfig(
                    **_dpo_training_kwargs(),
                    warmup_ratio               = 0.1,
                    num_train_epochs           = 1,
                    max_steps                  = -1,
                    learning_rate              = 5e-5,
                    fp16                       = False,
                    bf16                       = True,
                    logging_steps              = 10,
                    save_strategy              = "epoch",
                    save_total_limit           = 2,
                    optim                      = "adamw_8bit",
                    weight_decay               = 0.01,
                    lr_scheduler_type          = "cosine",
                    seed                       = 3407 + fold,
                    output_dir                 = fold_output_dir,
                    beta                       = 0.1,
                    max_prompt_length          = 2048,
                    max_length                 = 4096,
                ),
            )
        else:
            print(f"\n  Starting fold {fold} SFT (Qwen3-32B 4bit, A100 80GB, 1 epoch)...")
            def formatting_prompts_func(examples):
                texts = []
                for instruction, output in zip(examples["instruction"], examples["output"]):
                    texts.append(f"### Instruction:\n{instruction}\n\n### Response:\n{output}")
                return {"text": texts}
            dataset = dataset.map(formatting_prompts_func, batched=True)

            trainer = SFTTrainer(
                model            = model,
                processing_class = tokenizer,
                train_dataset    = dataset,
                args = SFTConfig(
                    **_sft_training_kwargs(),
                    dataset_text_field         = "text",
                    max_seq_length             = max_seq_length,
                    dataset_num_proc           = SFT_DATASET_NUM_PROC,
                    padding_free               = True,
                    warmup_ratio               = 0.1,
                    num_train_epochs           = 1,
                    max_steps                  = -1,
                    learning_rate              = 1e-4,
                    fp16                       = False,
                    bf16                       = True,
                    logging_steps              = 20,
                    save_strategy              = "epoch",
                    save_total_limit           = 2,
                    optim                      = "adamw_8bit",
                    weight_decay               = 0.01,
                    lr_scheduler_type          = "cosine",
                    seed                       = 3407 + fold,
                    output_dir                 = fold_output_dir,
                ),
            )

        trainer.train()

        fold_model_dir = os.path.join(base_dir, f"lora_model_fold_{fold}")
        model.save_pretrained(fold_model_dir)
        tokenizer.save_pretrained(fold_model_dir)
        print(f"  Fold {fold} saved to '{fold_model_dir}'")

        # 검증
        train_sites = set(train_fold["site_token"].astype(str))
        metrics = evaluate_full_pipeline(
            model,
            tokenizer,
            val_fold,
            retriever,
            train_sites,
            base_dir,
            max_seq_length,
            compact_prompt=e3_compact_prompt,
        )
        all_oof_metrics.append({"fold": fold, **metrics.get("overall", {})})

        # fold별 eval_metrics 별도 저장
        out_dir = os.path.join(base_dir, "outputs")
        os.makedirs(out_dir, exist_ok=True)
        with open(os.path.join(out_dir, f"eval_metrics_fold_{fold}.json"), "w") as f:
            json.dump(metrics, f, indent=2, ensure_ascii=False)

        # GPU 메모리 해제
        del model, tokenizer, trainer, dataset
        torch.cuda.empty_cache()
        print(f"  Fold {fold} complete. GPU memory cleared.")

    # OOF 전체 결과 요약
    print(f"\n{'='*60}")
    print("  OOF SUMMARY")
    print(f"{'='*60}")
    for m in all_oof_metrics:
        em = m.get('exact_match', 'N/A')
        print(f"  Fold {m['fold']}: EM={em}")

    out_dir = os.path.join(base_dir, "outputs")
    os.makedirs(out_dir, exist_ok=True)
    with open(os.path.join(out_dir, "oof_summary.json"), "w") as f:
        json.dump(all_oof_metrics, f, indent=2)
    print("OOF training complete.")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--oof", action="store_true", help="Run OOF training mode.")
    parser.add_argument(
        "--e2",
        action="store_true",
        help="E2: boost real_web augmentation in training data generation.",
    )
    parser.add_argument(
        "--e3",
        action="store_true",
        help="E3: use compact candidate formatting in prompts.",
    )
    args = parser.parse_args()

    if args.oof:
        train_oof(e2_realweb_boost=args.e2, e3_compact_prompt=args.e3)
    else:
        train(e2_realweb_boost=args.e2, e3_compact_prompt=args.e3)
'''

with open('/content/src/train.py', 'w', encoding='utf-8') as f:
    f.write(train_py)

inference_py = r'''# -*- coding: utf-8 -*-
import csv
import argparse
import json
import os
import re
from typing import Any

import pandas as pd
from tqdm import tqdm
from unsloth import FastLanguageModel

from collections import Counter

from preprocess import (
    enforce_consistency,
    extract_value_from_task,
    fallback_rule_based,
    detect_html_type,
    extract_workflow_context,
    format_numbered_candidates,
    choice_to_candidate_id,
    get_consistency_debug,
    build_prompt,
    rerank_candidates_by_embedding,
    RETRIEVAL_K,
)
from retrieval import ExampleRetriever


# ─────────────────────────────────────────────
# 런타임 설정
# ─────────────────────────────────────────────
MAX_SEQ_LENGTH        = 4096
BATCH_SIZE            = 128     # 80GB VRAM Monster Pass
CSV_CHUNK_SIZE        = 512     # 청크 단위 상향
USE_RETRIEVAL         = True
USE_CONSISTENCY       = True
OOF_N_FOLDS           = 3
EXPERIMENT_E3         = True    # 32B는 compact prompt가 효율적

# ─────────────────────────────────────────────
# 추론 통계
# ─────────────────────────────────────────────
STATS = Counter()


def reset_stats():
    STATS.clear()


def get_stats():
    return dict(STATS)


# ─────────────────────────────────────────────
# JSON 파싱
# ─────────────────────────────────────────────

def _parse_json_safe(text: str) -> dict | None:
    """중괄호가 중첩되지 않는 마지막 JSON 블록부터 역순으로 파싱 시도.

    모델이 thinking 도중 { } 를 사용해도 맨 끝 JSON만 파싱되도록 보장.
    """
    candidates_json = list(re.finditer(r'\{[^{}]*\}', text, re.DOTALL))
    for m in reversed(candidates_json):
        try:
            return json.loads(m.group(0))
        except json.JSONDecodeError:
            continue
    return None


def extract_json_answers(responses: list[str],
                         candidates_list: list[list[dict]]) -> list[tuple[str, str, str]]:
    """LLM 응답 배치를 파싱하고 choice 번호를 candidate_id로 변환한다."""
    results = []
    for resp, candidates in zip(responses, candidates_list):
        try:
            # Qwen3 thinking: </think> 이후만 파싱 (없으면 전체에서 역순 탐색)
            search_text = resp.split('</think>', 1)[-1] if '</think>' in resp else resp
            data = _parse_json_safe(search_text)
            if data is None:
                results.append(("CLICK", "", ""))
                continue
            op    = str(data.get("op", "CLICK")).upper()
            if op not in ("CLICK", "TYPE", "SELECT"):
                op = "CLICK"
            value     = "" if op == "CLICK" else str(data.get("value", ""))
            choice    = data.get("choice", "")
            target_id = choice_to_candidate_id(choice, candidates)
            results.append((op, target_id, value))
        except Exception:
            results.append(("CLICK", "", ""))
    return results


# ─────────────────────────────────────────────
# 제출 레코드 정리
# ─────────────────────────────────────────────

def _clean_submission_record(q_id, result: dict[str, Any]) -> dict[str, str]:
    op = str(result.get("op", "CLICK"))
    if op not in {"CLICK", "TYPE", "SELECT"}:
        op = "CLICK"
    value = str(result.get("value", "")).replace("\n", " ").replace("\r", " ").strip()
    if op == "CLICK":
        value = ""
    return {
        "id":        q_id,
        "op":        op,
        "target_id": str(result.get("target_id", "")),
        "value":     value,
    }


# ─────────────────────────────────────────────
# LLM 배치 추론
# ─────────────────────────────────────────────

def _run_llm_batch(batch: list[dict[str, Any]], model, tokenizer) -> list[dict[str, str]]:
    if not batch:
        return []

    texts  = [item["text"] for item in batch]
    inputs = tokenizer(
        texts,
        return_tensors  = "pt",
        padding         = True,
        truncation      = True,
        max_length      = MAX_SEQ_LENGTH,
    ).to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens  = 512,    # Qwen3 thinking + CoT + JSON
        use_cache       = True,
        do_sample       = False,
        pad_token_id    = tokenizer.eos_token_id,
    )

    generated_ids = outputs[:, inputs["input_ids"].shape[1]:]
    responses     = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
    answers       = extract_json_answers(responses, [item["candidates"] for item in batch])

    records = []
    for item, (op, target_id, value), raw_resp in zip(batch, answers, responses):
        candidates = item["candidates"]
        valid_ids  = {str(c.get("candidate_id", "")) for c in candidates}
        row        = item["row"]

        STATS["total"] += 1
        if "</think>" in raw_resp:
            STATS["thinking_used"] += 1

        if str(target_id) not in valid_ids:
            # choice 변환 실패 → rule-based fallback
            pred = fallback_rule_based(row, candidates)
            STATS["llm_fallback_invalid_target"] += 1
        else:
            matched = next(
                (c for c in candidates if str(c.get("candidate_id", "")) == str(target_id)),
                None,
            )
            attrs = str(matched.get("attrs", "")) if matched else ""

            # LLM이 생성한 value 우선, 비어있으면 rule 보조
            if value and op != "CLICK":
                final_value = value
            elif op != "CLICK":
                final_value = extract_value_from_task(row["task"], op, attrs)
            else:
                final_value = ""

            pred = {"op": op, "target_id": target_id, "value": final_value}
            STATS["llm_success"] += 1

        pred["_task"] = row["task"]
        if USE_CONSISTENCY:
            pred = enforce_consistency(pred, candidates)
        records.append(_clean_submission_record(item["id"], pred))
    return records


# ─────────────────────────────────────────────
# 메인
# ─────────────────────────────────────────────

def _save_report(base_dir: str, final_sub):
    import json
    from datetime import datetime

    stats     = get_stats()
    guard     = get_consistency_debug()
    total     = stats.get("total", 1)
    llm_ok    = stats.get("llm_success", 0)
    llm_fb    = stats.get("llm_fallback_invalid_target", 0)
    think     = stats.get("thinking_used", 0)
    rule_only = stats.get("no_model_rule_only", 0)

    report = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M"),
        "total_rows": total,
        "source": {
            "llm_success":              {"n": llm_ok,    "pct": f"{llm_ok/total:.1%}"},
            "llm_fallback_bad_target":  {"n": llm_fb,    "pct": f"{llm_fb/total:.1%}"},
            "no_model_rule_only":       {"n": rule_only, "pct": f"{rule_only/total:.1%}"},
        },
        "thinking_used": {"n": think, "pct": f"{think/max(llm_ok+llm_fb,1):.1%}"},
        "consistency_guard": {
            k: {"n": v, "pct": f"{v/total:.1%}"} for k, v in sorted(guard.items())
        },
        "op_dist": dict(final_sub["op"].value_counts()),
    }

    # 콘솔 출력
    print("\n" + "="*55)
    print("  INFERENCE REPORT")
    print("="*55)
    print(f"  Total rows        : {total}")
    print(f"  LLM success       : {llm_ok:>5}  ({llm_ok/total:.1%})")
    print(f"  LLM fallback      : {llm_fb:>5}  ({llm_fb/total:.1%})  ← target 무효")
    print(f"  Rule-only (no LLM): {rule_only:>5}  ({rule_only/total:.1%})")
    print(f"  Thinking used     : {think:>5}  ({think/max(llm_ok+llm_fb,1):.1%})  ← <think> 토큰 확인")
    print("-"*55)
    print("  Consistency Guard repairs:")
    if guard:
        for k, v in sorted(guard.items(), key=lambda x: -x[1]):
            print(f"    {k:<35}: {v:>4}  ({v/total:.1%})")
    else:
        print("    (없음)")
    print("="*55 + "\n")

    # JSON 저장
    artifacts_dir = os.path.join(base_dir, "artifacts")
    os.makedirs(artifacts_dir, exist_ok=True)
    report_path = os.path.join(artifacts_dir, "inference_report.json")
    with open(report_path, "w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2)
    print(f"Report saved: {report_path}")


def main(e3_compact_prompt: bool = EXPERIMENT_E3):
    global EXPERIMENT_E3
    EXPERIMENT_E3 = bool(e3_compact_prompt)
    base_dir        = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
    train_path      = os.path.join(base_dir, "data", "train.csv")
    test_path       = os.path.join(base_dir, "data", "test.csv")
    sample_sub_path = os.path.join(base_dir, "data", "somenna_submission.csv")

    print("1. Preparing retriever from train data...")
    train_df  = pd.read_csv(train_path)
    retriever = ExampleRetriever().build(train_df) if USE_RETRIEVAL else None

    print("2. Loading LLM (LoRA bundle)...")
    lora_dir = os.path.join(base_dir, "lora_model")
    model, tokenizer = None, None
    if os.path.exists(lora_dir):
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name    = lora_dir,
            max_seq_length = MAX_SEQ_LENGTH,
            load_in_4bit  = True,
        )
        FastLanguageModel.for_inference(model)
        tokenizer.padding_side   = "left"
        tokenizer.truncation_side = "left"
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
    else:
        print("  lora_model not found — using rule-based fallback only")

    print("3. Running streaming inference (Dual-flow)...")
    final_results    = []
    llm_batch        = []

    def flush_llm_batch():
        nonlocal llm_batch
        if model is None or not llm_batch:
            return
        final_results.extend(_run_llm_batch(llm_batch, model, tokenizer))
        llm_batch = []

    for chunk in tqdm(pd.read_csv(test_path, chunksize=CSV_CHUNK_SIZE), desc="Chunks"):
        id_col = "id" if "id" in chunk.columns else chunk.columns[0]
        for _, row in chunk.iterrows():
            q_id = row[id_col]
            try:
                candidates = json.loads(row["candidate_elements"])
            except Exception:
                candidates = []

            if model is None:
                pred = fallback_rule_based(row, candidates)
                pred["_task"] = row["task"]
                if USE_CONSISTENCY:
                    pred = enforce_consistency(pred, candidates)
                final_results.append(_clean_submission_record(q_id, pred))
                STATS["total"] += 1
                STATS["no_model_rule_only"] += 1
                continue

            html_type = detect_html_type(row)

            # [Flow A] Workflow: RAG + Progress Status
            if html_type == "workflow":
                prompt = build_prompt(
                    row,
                    candidates,
                    retriever=retriever,
                    k=RETRIEVAL_K,
                    compact_candidates=EXPERIMENT_E3,
                )
                llm_batch.append({
                    "id":         q_id,
                    "text":       f"### Instruction:\n{prompt}\n\n### Response:\n<think>\n",
                    "candidates": candidates,
                    "row":        row,
                })

            # [Flow B] Real Web: Rerank + Intensive Thinking Mode
            else:
                # Universal Reranking is now inside build_prompt, 
                # but we can apply additional logic here if needed.
                prompt = build_prompt(
                    row,
                    candidates,
                    retriever=retriever,
                    k=RETRIEVAL_K,
                    compact_candidates=EXPERIMENT_E3,
                )
                llm_batch.append({
                    "id":         q_id,
                    "text":       f"### Instruction:\n{prompt}\n\n### Response:\n<think>\n",
                    "candidates": candidates,
                    "row":        row,
                })

            if len(llm_batch) >= BATCH_SIZE:
                flush_llm_batch()

    flush_llm_batch()

    print("4. Saving submission...")
    sub_df = pd.DataFrame(final_results)
    if os.path.exists(sample_sub_path):
        sample_sub = pd.read_csv(sample_sub_path)
        final_sub  = sample_sub[["id"]].merge(sub_df, on="id", how="left")
    else:
        final_sub = sub_df

    final_sub.fillna("", inplace=True)
    final_sub.loc[final_sub["op"] == "CLICK", "value"] = ""
    final_sub.loc[~final_sub["op"].isin(["CLICK", "TYPE", "SELECT"]), "op"] = "CLICK"

    empty_count = (final_sub["target_id"].astype(str).str.strip() == "").sum()
    if empty_count:
        print(f"Warning: {empty_count} rows with empty target_id")

    out_path = os.path.join(base_dir, "submission.csv")
    final_sub.to_csv(out_path, index=False, quoting=csv.QUOTE_NONNUMERIC)
    print(f"Saved: {out_path}  ({len(final_sub)} rows)")
    print(f"op dist: {dict(final_sub['op'].value_counts())}")

    _save_report(base_dir, final_sub)



def ensemble_main(e3_compact_prompt: bool = False):
    """3-Fold OOF 앙상블 추론: 각 fold 모델의 예측을 다수결로 결합."""
    from collections import Counter
    import torch
    global EXPERIMENT_E3
    EXPERIMENT_E3 = bool(e3_compact_prompt)

    base_dir        = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
    train_path      = os.path.join(base_dir, "data", "train.csv")
    test_path       = os.path.join(base_dir, "data", "test.csv")
    sample_sub_path = os.path.join(base_dir, "data", "somenna_submission.csv")

    print("1. Preparing retriever from train data...")
    train_df  = pd.read_csv(train_path)
    retriever = ExampleRetriever().build(train_df) if USE_RETRIEVAL else None

    # test 행 미리 로드
    test_df = pd.read_csv(test_path)
    id_col  = "id" if "id" in test_df.columns else test_df.columns[0]
    all_ids = test_df[id_col].tolist()

    fold_predictions = []

    for fold in range(OOF_N_FOLDS):
        lora_dir = os.path.join(base_dir, f"lora_model_fold_{fold}")
        if not os.path.exists(lora_dir):
            print(f"  Fold {fold} model not found at {lora_dir}, skipping")
            continue

        print(f"\n2-{fold}. Loading fold {fold} LoRA bundle...")
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name     = lora_dir,
            max_seq_length = MAX_SEQ_LENGTH,
            load_in_4bit   = True,
        )
        FastLanguageModel.for_inference(model)
        tokenizer.padding_side    = "left"
        tokenizer.truncation_side = "left"
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        print(f"  Running inference for fold {fold}...")
        fold_results = []
        llm_batch    = []

        def flush_batch():
            nonlocal llm_batch
            if not llm_batch:
                return
            fold_results.extend(_run_llm_batch(llm_batch, model, tokenizer))
            llm_batch = []

        for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc=f"Fold {fold}"):
            q_id = row[id_col]
            try:
                candidates = json.loads(row["candidate_elements"])
            except Exception:
                candidates = []

            html_type = detect_html_type(row)

            # Dual-flow logic preserved in ensemble
            prompt = build_prompt(
                row,
                candidates,
                retriever=retriever,
                k=RETRIEVAL_K,
                compact_candidates=EXPERIMENT_E3,
            )
            llm_batch.append({
                "id":         q_id,
                "text":       f"### Instruction:\n{prompt}\n\n### Response:\n<think>\n",
                "candidates": candidates,
                "row":        row,
            })
            if len(llm_batch) >= BATCH_SIZE:
                flush_batch()

        flush_batch()

        # dict로 변환
        fold_dict = {}
        for rec in fold_results:
            fold_dict[rec["id"]] = {
                "op": rec["op"], "target_id": rec["target_id"], "value": rec["value"]
            }
        fold_predictions.append(fold_dict)
        print(f"  Fold {fold}: {len(fold_dict)} predictions")

        # GPU 메모리 해제
        del model, tokenizer
        torch.cuda.empty_cache()

    # 다수결 앙상블
    print(f"\n3. Majority vote ensemble ({len(fold_predictions)} folds)...")
    final_results   = []
    agreement_count = 0

    for q_id in all_ids:
        votes = []
        for fp in fold_predictions:
            if q_id in fp:
                v = fp[q_id]
                votes.append((v["op"], v["target_id"], v["value"]))

        if not votes:
            final_results.append({"id": q_id, "op": "CLICK", "target_id": "", "value": ""})
            continue

        winner = Counter(votes).most_common(1)[0]
        if winner[1] == len(fold_predictions):
            agreement_count += 1
        op, target_id, value = winner[0]
        final_results.append({
            "id": q_id, "op": op, "target_id": target_id, "value": value
        })

    print(f"  만장일치 비율: {agreement_count}/{len(all_ids)} "
          f"({agreement_count/max(len(all_ids),1):.1%})")

    # 저장
    print("4. Saving submission...")
    sub_df = pd.DataFrame(final_results)
    if os.path.exists(sample_sub_path):
        sample_sub = pd.read_csv(sample_sub_path)
        final_sub  = sample_sub[["id"]].merge(sub_df, on="id", how="left")
    else:
        final_sub = sub_df

    final_sub.fillna("", inplace=True)
    final_sub.loc[final_sub["op"] == "CLICK", "value"] = ""
    final_sub.loc[~final_sub["op"].isin(["CLICK", "TYPE", "SELECT"]), "op"] = "CLICK"

    empty_count = (final_sub["target_id"].astype(str).str.strip() == "").sum()
    if empty_count:
        print(f"Warning: {empty_count} rows with empty target_id")

    out_path = os.path.join(base_dir, "submission.csv")
    final_sub.to_csv(out_path, index=False, quoting=csv.QUOTE_NONNUMERIC)
    print(f"Saved: {out_path}  ({len(final_sub)} rows)")
    print(f"op dist: {dict(final_sub['op'].value_counts())}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--ensemble", action="store_true", help="Run OOF ensemble inference mode.")
    parser.add_argument(
        "--e1",
        action="store_true",
        help="E1: for no-survivor tournament rows, retry once with op-fixed constraint.",
    )
    parser.add_argument(
        "--e3",
        action="store_true",
        help="E3: use compact candidate formatting in prompts.",
    )
    args = parser.parse_args()

    if args.ensemble:
        ensemble_main(e3_compact_prompt=args.e3)
    else:
        main(e1_no_survivor_retry=args.e1, e3_compact_prompt=args.e3)
'''

with open('/content/src/inference.py', 'w', encoding='utf-8') as f:
    f.write(inference_py)

retrieval_py = r'''import json
import re
from collections import defaultdict

import pandas as pd

from preprocess import get_history_signature, parse_attrs_str, detect_html_type


def _tokens(text):
    return set(re.findall(r'\w+', str(text).lower()))


def _jaccard(a, b):
    a_tokens, b_tokens = _tokens(a), _tokens(b)
    if not a_tokens or not b_tokens:
        return 0.0
    return len(a_tokens & b_tokens) / len(a_tokens | b_tokens)


def _candidate_label(candidate):
    if not candidate:
        return ""
    text = str(candidate.get('text', '')).strip()
    if text:
        return text
    attrs = parse_attrs_str(str(candidate.get('attrs', '')))
    return str(attrs.get('label', '') or attrs.get('aria-label', '') or attrs.get('placeholder', '') or attrs.get('name', '')).strip()


class ExampleRetriever:
    def __init__(self):
        self.examples = []
        # html_type-aware 인덱스 (real_web / workflow 오염 방지)
        self.by_site_sig  = defaultdict(list)   # (site, sig, html_type) → positions
        self.by_site      = defaultdict(list)   # (site, html_type) → positions
        self.by_sig       = defaultdict(list)   # (sig, html_type) → positions
        # html_type 무관 fallback 인덱스
        self.by_site_sig_any = defaultdict(list)
        self.by_site_any     = defaultdict(list)

    def build(self, train_df):
        self.examples        = []
        self.by_site_sig     = defaultdict(list)
        self.by_site         = defaultdict(list)
        self.by_sig          = defaultdict(list)
        self.by_site_sig_any = defaultdict(list)
        self.by_site_any     = defaultdict(list)

        for idx, row in train_df.iterrows():
            try:
                candidates = json.loads(row['candidate_elements'])
            except Exception:
                candidates = []

            target_id = str(row.get('target_id', ''))
            target = next((c for c in candidates if str(c.get('candidate_id', '')) == target_id), None)
            label = _candidate_label(target)
            value = "" if pd.isna(row.get('value', '')) else str(row.get('value', ''))
            if row.get('op') == 'CLICK':
                value = ""

            html_type = detect_html_type(row)
            example = {
                'id': str(row.get('id', idx)),
                'site_token': str(row.get('site_token', '')),
                'history_signature': get_history_signature(row.get('history', '')),
                'html_type': html_type,
                'task': str(row.get('task', '')),
                'history': "" if pd.isna(row.get('history', '')) else str(row.get('history', '')),
                'target_label': label,
                'target_op': str(row.get('op', 'CLICK')),
                'target_value': value,
            }
            pos = len(self.examples)
            self.examples.append(example)

            site = example['site_token']
            sig  = example['history_signature']

            # html_type-aware 인덱스
            self.by_site_sig[(site, sig, html_type)].append(pos)
            self.by_site[(site, html_type)].append(pos)
            self.by_sig[(sig, html_type)].append(pos)
            # html_type 무관 fallback
            self.by_site_sig_any[(site, sig)].append(pos)
            self.by_site_any[site].append(pos)
        return self

    def _rank(self, indexes, row, k, exclude_id=None):
        task = str(row.get('task', ''))
        scored = []
        for idx in indexes:
            ex = self.examples[idx]
            if exclude_id is not None and str(ex.get('id')) == str(exclude_id):
                continue
            scored.append((_jaccard(task, ex['task']), idx))
        scored.sort(key=lambda x: x[0], reverse=True)
        return [self.examples[idx] for _, idx in scored[:k]]

    def query(self, row, k=3, exclude_id=None):
        site      = str(row.get('site_token', ''))
        sig       = get_history_signature(row.get('history', ''))
        html_type = detect_html_type(row)

        # 1순위: 같은 사이트 + 같은 히스토리 패턴 + 같은 html_type
        results = self._rank(self.by_site_sig.get((site, sig, html_type), []), row, k, exclude_id)
        seen = {ex['id'] for ex in results}

        # 2순위: 같은 사이트 + 같은 html_type
        if len(results) < k:
            more = self._rank(self.by_site.get((site, html_type), []), row, k * 3, exclude_id)
            for ex in more:
                if ex['id'] not in seen:
                    results.append(ex)
                    seen.add(ex['id'])
                if len(results) >= k:
                    break

        # 3순위: 같은 히스토리 패턴 + 같은 html_type
        if len(results) < k:
            more = self._rank(self.by_sig.get((sig, html_type), []), row, k * 3, exclude_id)
            for ex in more:
                if ex['id'] not in seen:
                    results.append(ex)
                    seen.add(ex['id'])
                if len(results) >= k:
                    break

        # 4순위 (fallback): html_type 무관, 같은 사이트+히스토리 패턴
        if len(results) < k:
            more = self._rank(self.by_site_sig_any.get((site, sig), []), row, k * 3, exclude_id)
            for ex in more:
                if ex['id'] not in seen:
                    results.append(ex)
                    seen.add(ex['id'])
                if len(results) >= k:
                    break

        # 5순위 (최종 fallback): html_type 무관, 같은 사이트
        if len(results) < k:
            more = self._rank(self.by_site_any.get(site, []), row, k * 3, exclude_id)
            for ex in more:
                if ex['id'] not in seen:
                    results.append(ex)
                    seen.add(ex['id'])
                if len(results) >= k:
                    break

        return results[:k]


'''

with open('/content/src/retrieval.py', 'w', encoding='utf-8') as f:
    f.write(retrieval_py)

print("✅ All source files (preprocess, train, inference, retrieval) generated.")


## 4. 데이터 업로드
왼쪽 파일 탐색기에서 `/content/data/` 폴더에 `train.csv`, `test.csv`, `somenna_submission.csv` 파일을 업로드해 주세요.

## 5. 3-Fold OOF 학습 시작

In [ ]:
%cd /content
!python src/train.py --oof

## 6. 앙상블 추론 및 결과 다운로드

In [ ]:
%cd /content
!python src/inference.py --ensemble

from google.colab import files
if os.path.exists('submission.csv'):
    files.download('submission.csv')